In [ ]:
import json, os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec

load_dotenv()
oa = OpenAI()
pc = Pinecone()

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CORPUS = ROOT / "corpus"
EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"   # if OpenAI has retired it, swap in a current low-cost model from their models page
INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "checkpoint-rag")
NAMESPACE = "day2"           # Day 2 experiments stay separate from the real ingest later
print(ROOT)

In [ ]:
test = json.loads((ROOT / "evals" / "test_set.json").read_text(encoding="utf-8"))
q = test["queries"][0]
print("Q:", q["query"])
print("Expected sources:", q["expected_sources"])

In [ ]:
expected = [s for s in q["expected_sources"] if s.endswith(".md")]
all_md = sorted(p.relative_to(CORPUS).as_posix() for p in (CORPUS / "product-docs").rglob("*.md"))
distractors = [s for s in all_md if s not in expected][: 5 - len(expected)]
chosen = expected + distractors

docs = [{"source": s, "text": (CORPUS / s).read_text(encoding="utf-8")} for s in chosen]
for d in docs:
    print(f"{len(d['text']):>6} chars | {d['source']}")

assert all(len(d["text"]) < 30_000 for d in docs), "A doc is too big for Pinecone metadata. Tell Claude."

In [ ]:
resp = oa.embeddings.create(model=EMBED_MODEL, input=[d["text"] for d in docs])
for d, item in zip(docs, resp.data):
    d["vector"] = item.embedding
print(len(docs[0]["vector"]), "dims |", resp.usage.total_tokens, "tokens billed")

In [ ]:
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME, dimension=1536, metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
index = pc.Index(INDEX_NAME)

records = [
    {
        "id": d["source"],   # stable ID: re-running this cell overwrites, never duplicates
        "values": d["vector"],
        "metadata": {"source": d["source"], "doc_type": d["source"].split("/")[0], "text": d["text"]},
    }
    for d in docs
]
index.upsert(vectors=records, namespace=NAMESPACE)
print("Upserted", len(records))

In [ ]:
q_vec = oa.embeddings.create(model=EMBED_MODEL, input=q["query"]).data[0].embedding
res = index.query(vector=q_vec, top_k=3, include_metadata=True, namespace=NAMESPACE)

for m in res.matches:
    hit = "✅" if m.id in q["expected_sources"] else "  "
    print(f"{hit} {m.score:.3f} | {m.id}")

In [ ]:
context = "\n\n---\n\n".join(
    f"[Source: {m.metadata['source']}]\n{m.metadata['text']}" for m in res.matches
)
messages = [
    {"role": "system", "content": (
        "You answer questions about the Helix product using ONLY the sources provided. "
        "Cite the source path for each claim. "
        "If the sources don't contain the answer, say you don't know."
    )},
    {"role": "user", "content": f"Sources:\n{context}\n\nQuestion: {q['query']}"},
]
out = oa.chat.completions.create(model=CHAT_MODEL, messages=messages, temperature=0)

print(out.choices[0].message.content)
print("\nTokens:", out.usage.prompt_tokens, "in /", out.usage.completion_tokens, "out")
print("\nGround truth:", q["ground_truth_answer"])